# 02 — LunarLander environment factory and wrappers

**Goal.** Construct the two LunarLander variants used by the baselines and verify the two custom wrappers — `ActuatorFaultWrapper` and `DomainRandomizationWrapper` — behave as the thesis requires.

## Why these wrappers matter

The research question is *can a learning agent detect a change in its own actuator response*. To pose that question empirically we need:

1. A controlled way to **inject** an actuator fault at a known time step → `ActuatorFaultWrapper`.
2. A controlled way to **randomise** physics each episode to train a robustness baseline (PPO+DR) → `DomainRandomizationWrapper`.

## Mathematical formulation

**Actuator step-fault (continuous control):**

$$ a_\text{applied}(t) = M(t) \odot a_\text{commanded}(t), \qquad M(t) = \begin{cases} \mathbf{1} & t < t_\text{fault} \\\\ g & t \geq t_\text{fault} \end{cases} $$

where $g \in [0,1]^{|\mathcal{A}|}$ is a per-thruster gain vector. $g=\mathbf{0}$ is total loss, $g=\tfrac{1}{2}\mathbf{1}$ is 50 % thrust degradation.

**Domain randomisation (per episode $k$):**

$$ \xi_k \sim \mathcal{U}(\xi_\text{lo}, \xi_\text{hi}), \quad \xi \in \{g_\text{grav}, P_\text{main}, P_\text{side}, w, \tau\} $$

---

## Bug log

| # | Symptom | Diagnosis | Fix |
|---|---|---|---|
| 1 | `AttributeError: can't set attribute 'spec'` inside `ActuatorFaultWrapper.__init__` | `gym.Env` already exposes `spec` as a property (the `EnvSpec`). Assigning `self.spec = ...` collides with it. | Renamed instance fields to `self.fault_spec` and `self.dr_spec`. |
| 2 | `MAIN_ENGINE_POWER` override silently ignored across workers | The constant is a **module-level** attribute on `gymnasium.envs.box2d.lunar_lander`; with `DummyVecEnv` (single process) the latest reset wins. | Always use `SubprocVecEnv` for DR — each worker has its own module copy. |
| 3 | NaN observations after fault injection | Continuous action clipped *before* the gain was applied, so commanded ≠ clipped. | Apply `np.clip` to `a_applied` (post-gain), not to `a_commanded`. |
| 4 | `TypeError: 'numpy.int64' object is not iterable` from `vec.step(vec.action_space.sample())` | SB3's `VecEnv.action_space` is the **single-env** action space, not vectorised — `.sample()` returns one scalar. `SubprocVecEnv.step()` needs an iterable of `num_envs` actions. | Batch-sample: `np.array([vec.action_space.sample() for _ in range(vec.num_envs)])`. |


## 2.1 Make sure `src` is importable

In [1]:
import sys, pathlib
PY_ROOT = pathlib.Path('..').resolve() / 'py'
if str(PY_ROOT) not in sys.path:
    sys.path.insert(0, str(PY_ROOT))

## 2.2 Sanity-check both LunarLander variants

We use **v3** (Gymnasium 1.0) for both discrete and continuous flavours.

In [2]:
import gymnasium as gym

for env_id in ('LunarLander-v3', 'LunarLanderContinuous-v3'):
    env = gym.make(env_id)
    obs, info = env.reset(seed=0)
    print(f'{env_id:30s}  obs.shape={obs.shape}  action_space={env.action_space}')
    env.close()

LunarLander-v3                  obs.shape=(8,)  action_space=Discrete(4)
LunarLanderContinuous-v3        obs.shape=(8,)  action_space=Box(-1.0, 1.0, (2,), float32)


## 2.3 ActuatorFaultWrapper — discrete variant

**Behaviour to verify.** Before `fault_step`, commanded action passes through. After `fault_step`, any commanded action equal to `discrete_drop_action` is rewritten to `0` (no-op).

We push *main engine* (action `2`) at every step. After the fault, those commands should become no-ops and the lander should fall.

In [3]:
from src.envs.wrappers import ActuatorFaultSpec, ActuatorFaultWrapper

env = gym.make('LunarLander-v3')
env = ActuatorFaultWrapper(env, ActuatorFaultSpec(fault_step=50, discrete_drop_action=2))

obs, _ = env.reset(seed=0)
rewards_pre, rewards_post = [], []
for t in range(120):
    obs, r, term, trunc, _ = env.step(2)        # always fire main engine
    (rewards_pre if t < 50 else rewards_post).append(r)
    if term or trunc:
        break
env.close()

import numpy as np
print(f'pre-fault  mean r = {np.mean(rewards_pre):+.3f}')
print(f'post-fault mean r = {np.mean(rewards_post):+.3f}')

pre-fault  mean r = -1.837
post-fault mean r = -2.526


## 2.4 ActuatorFaultWrapper — continuous variant

Set the second thruster (lateral) to zero gain after step 80. The lander loses lateral control.

In [4]:
import numpy as np
from src.envs.wrappers import ActuatorFaultSpec, ActuatorFaultWrapper

env = gym.make('LunarLanderContinuous-v3')
env = ActuatorFaultWrapper(
    env,
    ActuatorFaultSpec(fault_step=80, fault_gain=np.array([1.0, 0.0], dtype=np.float32)),
)
obs, _ = env.reset(seed=0)
for _ in range(200):
    a = env.action_space.sample()
    obs, r, term, trunc, _ = env.step(a)
    if term or trunc:
        break
env.close()
print('continuous fault wrapper: stepped without error.')

continuous fault wrapper: stepped without error.


## 2.5 DomainRandomizationWrapper

Verify that `env.unwrapped.gravity` changes across resets and that the per-episode samples lie inside the configured ranges.

In [5]:
from src.envs.wrappers import DomainRandomizationSpec, DomainRandomizationWrapper

spec = DomainRandomizationSpec(seed=0, log_per_episode=False)
env = gym.make('LunarLander-v3')
env = DomainRandomizationWrapper(env, spec)

samples = []
for _ in range(5):
    env.reset()
    samples.append(env.last_sample)
env.close()

import pandas as pd
df = pd.DataFrame(samples)
print(df.to_string(index=False))

# Assertions: gravity stays inside (-12, -8)
assert df['gravity'].between(-12, -8).all()
assert df['main_engine_power'].between(11, 17).all()
print('[ok] all samples lie inside the configured DR ranges.')

   gravity  main_engine_power  side_engine_power  wind_power  turbulence_power
 -9.452153          12.618720           0.416389    5.247915          1.719905
 -8.348978          14.639815           0.691799   13.154375          1.902609
 -8.736586          11.016431           0.742962    5.503784          1.594483
-11.297378          16.179074           0.616584    9.495678          1.134031
-11.886721          11.745700           0.668250   14.707843          1.423078
[ok] all samples lie inside the configured DR ranges.


## 2.6 Vec-env factory smoke test

`make_vec_env` returns a `SubprocVecEnv` (CPU-parallel). Stepping it four workers wide should produce a 4-row observation matrix.

> **Gotcha (bug #4 above).** `vec.action_space` is the *single-env* action space, not the vectorised one. We have to sample one action per worker and stack into a `(num_envs,)` array before calling `step`.

In [6]:
import numpy as np
from src.envs.lunarlander_factory import make_vec_env

vec = make_vec_env('LunarLander-v3', n_envs=4, base_seed=0)
obs = vec.reset()
print(f'obs.shape    = {obs.shape}    (expect (4, 8))')

# One action per worker — VecEnv.action_space is single-env, so we batch ourselves.
actions = np.array([vec.action_space.sample() for _ in range(vec.num_envs)])
obs, r, done, info = vec.step(actions)
print(f'step ok: actions.shape={actions.shape}  r.shape={r.shape}  done.shape={done.shape}')
vec.close()

obs.shape    = (4, 8)    (expect (4, 8))
step ok: actions.shape=(4,)  r.shape=(4,)  done.shape=(4,)


✅ **Checkpoint.** Wrappers and the vec-env factory work as specified. Proceed to **03 — PPO baseline**.